# Ejercicios 03. Señales reales: audio, imágenes y señales biomédicas

Antes de empezar:

1. Repasa el notebook de teoría `notebooks/03_cargar_ver_escuchar.ipynb`.
2. Ejecuta la celda de preparación, que es la primera celda de código.
3. Busca las marcas `COMPLETA` y cambia los `...` por tu código.
4. Ejecuta las celdas de verificación. Imprimen "Correcto" si tu resultado coincide y "Revisa" si no.
5. Contesta las preguntas de reflexión con tus propias palabras. Es la parte más importante.

Si una celda marca error, lee el último renglón del mensaje; casi siempre ahí está la explicación.
Por ejemplo, `NameError` quiere decir que usaste una variable que no existe o que no ejecutaste
una celda anterior.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## Ejercicio 1. Explorar un audio

Carga `../datos/mezcla_tonos.wav` y responde: a) $f_s$, b) número de muestras, c) duración,
d) instante de la muestra 5000, e) grafica los primeros **20 ms** con el eje en segundos, f) escúchalo.

In [ ]:
fs, x = ...                 # COMPLETA

N = ...                     # COMPLETA
duracion = ...              # COMPLETA
t_5000 = ...                # COMPLETA
print(f"fs = {fs} Hz, N = {N}, duración = {duracion} s, t[5000] = {t_5000} s")

n = np.arange(0, N)
t = ...                     # COMPLETA: eje de tiempo
cuantas = ...               # COMPLETA: muestras en 20 ms
pds.graficar_continua(t[:cuantas], x[:cuantas], titulo="Primeros 20 ms")
plt.show()

In [ ]:
pds.comprobar(fs == 8000, "fs = 8000 Hz")
pds.comprobar_iguales(duracion, 2.0, "duración = 2 s")
pds.comprobar_iguales(t_5000, 0.625, "la muestra 5000 está en t = 0.625 s")
pds.comprobar(cuantas == 160, "20 ms = 160 muestras")
pds.reproducir(x, fs)

**Reflexión:** ¿puedes saber "a ojo", viendo la gráfica en el tiempo, qué frecuencias tiene esta mezcla? (En el notebook 08 lo resolveremos con la FFT).

## Ejercicio 2. Velocidad y reversa

Con el tono de 440 Hz (`../datos/tono_440hz.wav`):
a) Reprodúcelo **al revés** (inversión en el tiempo). ¿Suena distinto? ¿Por qué?
b) Calcula a qué frecuencia en Hz se escuchará si lo reproduces con $f_s' = 1.5 f_s$ y compruébalo escuchándolo.

In [ ]:
from IPython.display import display

fs, x = pds.cargar_audio("../datos/tono_440hz.wav")

x_reves = ...                       # COMPLETA
display(pds.reproducir(x_reves, fs))

f_digital = ...                     # COMPLETA: frecuencia digital del tono
fs_nueva = 1.5 * fs
F_nueva = ...                       # COMPLETA: F = f · fs_nueva
print("Se escuchará a", F_nueva, "Hz")
display(pds.reproducir(x, fs_nueva))

In [ ]:
pds.comprobar_iguales(F_nueva, 660, "se escucha a 660 Hz (una quinta arriba)")

**Reflexión (a):** ¿por qué el tono al revés suena igual?

## Ejercicio 3. Una imagen tiene "frecuencias"

Carga `../datos/imagen_prueba.png` (en grises). Extrae como señales 1D: a) la **fila 75** y b) la **fila 120**.
Grafícalas con `stem` (sólo las columnas 140 a 200 de la fila 120). Luego estima el **periodo de las franjas en pixeles**.

In [ ]:
imagen = ...                         # COMPLETA
pds.mostrar_imagen(imagen, titulo="imagen de prueba")
plt.show()

fila75 = ...                         # COMPLETA
fila120 = ...                        # COMPLETA
columnas = np.arange(0, imagen.shape[1])

figura, ejes = plt.subplots(2, 1, figsize=(10, 5))
pds.graficar_continua(columnas, fila75, titulo="fila 75", etiqueta_x="columna", ax=ejes[0])
pds.graficar_discreta(columnas[140:200], fila120[140:200], titulo="fila 120, columnas 140-199",
                      etiqueta_x="columna", ax=ejes[1])
plt.tight_layout()
plt.show()

periodo_franjas = ...                # COMPLETA: cuenta en la gráfica cuántos pixeles hay entre dos máximos

In [ ]:
pds.comprobar(periodo_franjas == 8, "las franjas se repiten cada 8 pixeles (f = 1/8 ciclos/pixel)")

## Ejercicio 4. Frecuencia cardíaca a partir del ECG

Mide la frecuencia cardíaca del ECG **detectando los picos R** con un algoritmo sencillo:

Una muestra $i$ es un pico R si:
1. es mayor que un **umbral**,
2. es mayor que sus dos vecinas ($x[i-1]$ y $x[i+1]$), y
3. han pasado al menos **0.2 s** desde el último pico detectado (*periodo refractario*: el corazón no puede latir dos
   veces tan seguido). Sin esta regla, el ruido de 60 Hz crea **dos** máximos locales en la punta de algunos picos R.

Después: intervalos RR (en muestras) → en segundos → latidos por minuto: $\text{lpm} = 60 / \overline{RR}$.

In [ ]:
fs_ecg = 360
t_ecg, ecg = pds.cargar_senal_texto("../datos/ecg_sintetico.csv", fs=fs_ecg, columna=1)

umbral = 0.8                     # mV: observa la gráfica del notebook para elegirlo
refractario = ...                # COMPLETA: 0.2 s en muestras
picos = []                       # aquí guardaremos las POSICIONES de los picos R
ultimo_pico = -10000             # posición del último pico (al inicio, "hace mucho")
for i in range(1, len(ecg) - 1):
    if ...:                      # COMPLETA: las tres condiciones (usa "and")
        picos.append(i)
        ultimo_pico = i
picos = np.array(picos)
print("Picos encontrados:", len(picos))

# Intervalos RR
RR_muestras = ...                # COMPLETA (pista: np.diff(picos) resta cada pico del anterior)
RR_segundos = ...                # COMPLETA
lpm = ...                        # COMPLETA: 60 / promedio de RR_segundos
print(f"RR promedio = {np.mean(RR_segundos):.3f} s  ->  {lpm:.1f} latidos por minuto")

pds.graficar_continua(t_ecg, ecg, titulo="Picos R detectados", etiqueta_y="mV")
plt.plot(t_ecg[picos], ecg[picos], "rv")
plt.show()

In [ ]:
pds.comprobar(len(picos) == 13, "se detectan los 13 latidos")
pds.comprobar(abs(lpm - 75) < 1, "frecuencia cardíaca ≈ 75 lpm")

**Reflexión:** quita la condición del periodo refractario y vuelve a ejecutar: ¿cuántos picos detecta ahora y por qué? ¿Qué pasaría si la deriva de línea base fuera más grande (±1 mV)? ¿Qué propones?

## Ejercicio 5 (opcional). Tu propia señal

Graba tu voz diciendo una vocal ("aaaa") durante 2 s con el celular, conviértela a .wav (por ejemplo con Audacity),
guárdala en `datos/` y repite el ejercicio 1 con ella. Haz un acercamiento de 30 ms: ¿ves una forma que se repite?
Ese periodo es el de tu **tono de voz** (frecuencia fundamental).